# BEAM CORE Workflow Demo: Provenance Tracking with Consist

Three sequential models - land use, travel demand, network assignment - instrumented with
Consist for automatic provenance tracking, scenario comparison, and reproducible results.

**What this notebook shows:**
- How to instrument a multi-step workflow with minimal code changes
- How to compare two scenarios and see exactly what changed
- How to trace any output back through its full computation chain

## Questions this notebook answers

- "What exact parameters produced this VMT forecast I sent to the MPO six months ago?"
- "Did we run the 2045 scenario with the updated BPR parameters or the old ones?"
- "Which trip table fed the network model in this scenario?"


## Imports and Setup

Initialize the tracker and a clean run directory for this demo session.


In [1]:
from __future__ import annotations

import os
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import consist
from consist import CacheOptions, ExecutionOptions, Tracker


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (missing pyproject.toml)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
RUN_DIR = EXAMPLES_DIR / "runs" / "beam_core_demo"
SESSION_ID = os.getenv("CONSIST_SESSION_ID", "demo")
DB_PATH = RUN_DIR / f"beam_core_demo_{SESSION_ID}.duckdb"

RUN_DIR.mkdir(parents=True, exist_ok=True)
if DB_PATH.exists():
    DB_PATH.unlink()

tracker = Tracker(
    run_dir=RUN_DIR,
    db_path=DB_PATH,
    hashing_strategy="fast",
    project_root=str(RUN_DIR),
)

print(f"Using tracker DB: {DB_PATH}")

Using tracker DB: /Users/zaneedell/git/consist/examples/runs/beam_core_demo/beam_core_demo_demo.duckdb


## Configuration

Land use controls city growth, demand controls behavioral mode choice, and network controls congestion response.


In [2]:
@dataclass
class LandUseConfig:
    n_zones: int = 5
    growth_rate: float = 0.02


@dataclass
class DemandConfig:
    value_of_time: float = 15.0
    transit_fare: float = 2.50
    parking_cost_cbd: float = 10.0


@dataclass
class NetworkConfig:
    capacity_factor: float = 1.0
    speed_reduction_factor: float = 0.5


land_use_config = LandUseConfig()
demand_config = DemandConfig()
network_config = NetworkConfig()

print(f"LandUseConfig: {land_use_config}")
print(f"DemandConfig: {demand_config}")
print(f"NetworkConfig: {network_config}")

LandUseConfig: LandUseConfig(n_zones=5, growth_rate=0.02)
DemandConfig: DemandConfig(value_of_time=15.0, transit_fare=2.5, parking_cost_cbd=10.0)
NetworkConfig: NetworkConfig(capacity_factor=1.0, speed_reduction_factor=0.5)


## Model Functions

The model functions below are pure Python - no Consist calls inside them. Consist wraps them at the workflow level.


In [3]:
def run_land_use(config: LandUseConfig) -> pd.DataFrame:
    base_data = [
        {"zone_id": 1, "zone_type": "suburban", "population": 5000, "employment": 1000},
        {"zone_id": 2, "zone_type": "inner", "population": 8000, "employment": 3000},
        {"zone_id": 3, "zone_type": "cbd", "population": 2000, "employment": 15000},
        {"zone_id": 4, "zone_type": "inner", "population": 7000, "employment": 4000},
        {"zone_id": 5, "zone_type": "suburban", "population": 4000, "employment": 800},
    ]
    if config.n_zones != 5:
        raise ValueError("This demo uses exactly 5 zones.")

    land_use = pd.DataFrame(base_data)
    growth_multiplier = 1.0 + config.growth_rate
    land_use["population"] = (land_use["population"] * growth_multiplier).round(0)
    land_use["employment"] = (land_use["employment"] * growth_multiplier).round(0)
    land_use["residential_units"] = (land_use["population"] / 2.3).round(0)
    return land_use


def _normalize_mode_shares(
    car: float, transit: float, walk: float
) -> tuple[float, float, float]:
    total = car + transit + walk
    if total <= 0:
        return 0.0, 0.0, 0.0
    return car / total, transit / total, walk / total


def run_demand_model(land_use: pd.DataFrame, config: DemandConfig) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    zones = land_use[["zone_id", "zone_type", "population", "employment"]].copy()

    rows: list[dict[str, float | int | str]] = []
    for origin in zones.itertuples(index=False):
        for destination in zones.itertuples(index=False):
            if origin.zone_id == destination.zone_id:
                continue

            base_trips = (origin.population * destination.employment) / 100_000
            trip_noise = max(0.85, float(rng.normal(loc=1.0, scale=0.03)))
            base_trips *= trip_noise

            fare_penalty = max(0.0, config.transit_fare - 2.5) * 0.02
            vot_effect = (config.value_of_time - 15.0) * 0.002
            car_share = 0.60 + vot_effect
            transit_share = 0.25 - fare_penalty - (0.5 * vot_effect)
            walk_share = 0.15 + fare_penalty

            if destination.zone_type == "cbd":
                parking_shift = max(0.0, config.parking_cost_cbd - 10.0) * 0.015
                car_share -= parking_shift
                transit_share += parking_shift * 0.85
                walk_share += parking_shift * 0.15

            car_share, transit_share, walk_share = _normalize_mode_shares(
                max(car_share, 0.05),
                max(transit_share, 0.05),
                max(walk_share, 0.05),
            )

            distance_miles = abs(origin.zone_id - destination.zone_id) * 3.0
            for mode, share in (
                ("car", car_share),
                ("transit", transit_share),
                ("walk", walk_share),
            ):
                rows.append(
                    {
                        "origin_zone": int(origin.zone_id),
                        "dest_zone": int(destination.zone_id),
                        "mode": mode,
                        "trips": round(base_trips * share, 3),
                        "distance_miles": float(distance_miles),
                    }
                )

    return pd.DataFrame(rows)


def run_network_model(trip_table: pd.DataFrame, config: NetworkConfig) -> pd.DataFrame:
    car_trips = float(trip_table.loc[trip_table["mode"] == "car", "trips"].sum())
    transit_ridership = float(
        trip_table.loc[trip_table["mode"] == "transit", "trips"].sum()
    )
    total_trips = float(trip_table["trips"].sum())
    vmt = float(
        (
            trip_table.loc[trip_table["mode"] == "car", "trips"]
            * trip_table.loc[trip_table["mode"] == "car", "distance_miles"]
        ).sum()
    )

    capacity = 20_000.0 * config.capacity_factor
    volume_ratio = vmt / capacity if capacity > 0 else 0.0
    congestion_factor = 1.0 + 0.15 * (volume_ratio**4)

    free_flow_speed_mph = 35.0
    avg_network_speed_mph = free_flow_speed_mph / (
        1.0 + config.speed_reduction_factor * (congestion_factor - 1.0)
    )

    metrics = [
        {"metric": "vmt", "value": round(vmt, 3)},
        {"metric": "transit_ridership", "value": round(transit_ridership, 3)},
        {"metric": "total_trips", "value": round(total_trips, 3)},
        {"metric": "car_share", "value": round(car_trips / total_trips, 6)},
        {"metric": "avg_network_speed_mph", "value": round(avg_network_speed_mph, 3)},
        {"metric": "congestion_factor", "value": round(congestion_factor, 6)},
    ]
    return pd.DataFrame(metrics)

## Step Wrappers

Step wrappers handle provenance logging. The model functions stay unchanged.


In [4]:
def land_use_step(*, config: LandUseConfig) -> None:
    land_use = run_land_use(config)
    consist.log_dataframe(land_use, key="land_use")


def demand_step(*, land_use: pd.DataFrame, config: DemandConfig) -> None:
    trip_table = run_demand_model(land_use, config)
    consist.log_dataframe(trip_table, key="trip_table")


def network_step(*, trip_table: pd.DataFrame, config: NetworkConfig) -> None:
    network_performance = run_network_model(trip_table, config)
    consist.log_dataframe(network_performance, key="network_performance")

## run_scenario() Helper

Run the three-step workflow for a single scenario run id.


In [5]:
def run_scenario(
    land_use_config: LandUseConfig,
    demand_config: DemandConfig,
    network_config: NetworkConfig,
    scenario_run_id: str,
) -> dict[str, object]:
    cache_options = CacheOptions(
        validate_cached_outputs="lazy",
        cache_hydration="inputs-missing",
    )

    with consist.scenario(
        scenario_run_id,
        tracker=tracker,
        config={
            "land_use": asdict(land_use_config),
            "demand": asdict(demand_config),
            "network": asdict(network_config),
        },
        facet_from=["land_use", "demand", "network"],
        tags=["demo", "beam_core"],
    ) as scenario:
        land_use_result = scenario.run(
            name="land_use_model",
            fn=land_use_step,
            config=asdict(land_use_config),
            outputs=["land_use"],
            execution_options=ExecutionOptions(
                load_inputs=False,
                runtime_kwargs={"config": land_use_config},
            ),
            cache_options=cache_options,
        )

        demand_result = scenario.run(
            name="demand_model",
            fn=demand_step,
            config=asdict(demand_config),
            inputs={"land_use": consist.ref(land_use_result, key="land_use")},
            outputs=["trip_table"],
            execution_options=ExecutionOptions(
                load_inputs=True,
                runtime_kwargs={"config": demand_config},
            ),
            cache_options=cache_options,
        )

        network_result = scenario.run(
            name="network_model",
            fn=network_step,
            config=asdict(network_config),
            inputs={"trip_table": consist.ref(demand_result, key="trip_table")},
            outputs=["network_performance"],
            execution_options=ExecutionOptions(
                load_inputs=True,
                runtime_kwargs={"config": network_config},
            ),
            cache_options=cache_options,
        )

    return {
        "scenario_run_id": scenario_run_id,
        "land_use_result": land_use_result,
        "demand_result": demand_result,
        "network_result": network_result,
    }

## Run Baseline Scenario

Run the baseline assumptions end-to-end and inspect the final network metrics.


In [6]:
SCENARIO_NAME = "beam_core_demo"
base_run_id = f"{SCENARIO_NAME}_{SESSION_ID}_baseline"
baseline = run_scenario(land_use_config, demand_config, network_config, base_run_id)
BASELINE_RUN_ID = baseline["scenario_run_id"]
print("Baseline run complete.")

Baseline run complete.


In [7]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
if network_run is None:
    raise RuntimeError("Could not find baseline network_model run")

network_outputs = tracker.get_run_outputs(network_run.id)
consist.load_df(network_outputs["network_performance"], tracker=tracker)

,metric,value
0,vmt,15587.910000
1,transit_ridership,1366.224000
2,total_trips,5464.897000
3,car_share,0.600000
4,avg_network_speed_mph,34.057000
5,congestion_factor,1.055351


## Policy Scenario (Higher CBD Parking)

Now run a policy scenario with higher CBD parking costs. We expect mode shift away from car and toward transit.


In [8]:
high_parking_demand_config = DemandConfig(parking_cost_cbd=20.0)
policy_run_id = f"{SCENARIO_NAME}_{SESSION_ID}_high_parking"
policy = run_scenario(
    land_use_config,
    high_parking_demand_config,
    network_config,
    policy_run_id,
)
POLICY_RUN_ID = policy["scenario_run_id"]
print("Policy run complete.")

Policy run complete.


In [9]:
def _load_network_performance(scenario_run_id: str) -> pd.DataFrame:
    run = tracker.find_run(
        parent_id=scenario_run_id,
        model="network_model",
        status="completed",
    )
    if run is None:
        raise RuntimeError(f"Could not find network_model run for {scenario_run_id}")
    outputs = tracker.get_run_outputs(run.id)
    return consist.load_df(outputs["network_performance"], tracker=tracker)


baseline_perf = _load_network_performance(BASELINE_RUN_ID).rename(
    columns={"value": "baseline"}
)
policy_perf = _load_network_performance(POLICY_RUN_ID).rename(
    columns={"value": "policy"}
)

comparison = baseline_perf.merge(policy_perf, on="metric", how="inner")
comparison["delta"] = (comparison["policy"] - comparison["baseline"]).round(6)

key_metrics = ["vmt", "transit_ridership", "car_share"]
comparison = comparison[comparison["metric"].isin(key_metrics)].copy()
comparison["metric"] = pd.Categorical(comparison["metric"], key_metrics, ordered=True)
comparison.sort_values("metric").reset_index(drop=True)

,metric,baseline,policy,delta
0,vmt,15587.910,13292.57400,-2295.33600
1,transit_ridership,1366.224,1838.49600,472.27200
2,car_share,0.600,0.49833,-0.10167


## What Changed Between Scenarios? (DEMO MOMENT 1)

Consist stores the full config for every run. `diff_runs()` surfaces only the parameters that actually changed.


In [10]:
diff = tracker.diff_runs(BASELINE_RUN_ID, POLICY_RUN_ID)

diff_rows = []
for key in sorted(diff["changes"]):
    change = diff["changes"][key]
    if change["status"] == "same":
        continue
    diff_rows.append(
        {
            "parameter": key,
            "baseline": change.get("left"),
            "policy": change.get("right"),
        }
    )

pd.DataFrame(diff_rows)

,parameter,baseline,policy
0,demand.parking_cost_cbd,10.0,20.0


## How Was This Output Produced? (DEMO MOMENT 2)

Consist traces any output back through its full computation chain. The lineage tree shows exactly which step produced each artifact and what inputs it consumed.


In [11]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
if network_run is None:
    raise RuntimeError("Could not find baseline network_model run")

network_art = tracker.get_run_outputs(network_run.id)["network_performance"]
tracker.print_lineage(network_art.id, max_depth=6)

Lineage
└── network_performance (parquet)
    └── ← network_model
        └── trip_table (parquet)
            └── ← demand_model
                └── land_use (parquet)
                    └── ← land_use_model

## Which Parameters Produced This VMT Number? (DEMO MOMENT 3)

Six months later: "What parameters produced that VMT forecast?" One query.


In [12]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
demand_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="demand_model",
    status="completed",
)
if network_run is None or demand_run is None:
    raise RuntimeError("Missing baseline demand/network runs")

network_outputs = tracker.get_run_outputs(network_run.id)
network_perf = consist.load_df(network_outputs["network_performance"], tracker=tracker)
vmt_value = network_perf.loc[network_perf["metric"] == "vmt", "value"].iloc[0]

demand_config_stored = tracker.get_run_config(demand_run.id)

pd.DataFrame(
    [
        {
            "vmt": vmt_value,
            "network_performance_artifact": str(
                network_outputs["network_performance"].id
            ),
            "demand_model_run": demand_run.id,
            **{
                key: demand_config_stored.get(key)
                for key in ["value_of_time", "transit_fare", "parking_cost_cbd"]
            },
        }
    ]
)

,vmt,network_performance_artifact,demand_model_run,value_of_time,transit_fare,parking_cost_cbd
0,15587.91,de0507f3-723a-4e5c-a0c8-85e5ccd318b0,beam_core_demo_demo_baseline_demand_model_4f9b...,15.0,2.5,10.0


## Smart Caching: Only Re-Run What Changed

Re-running the baseline with identical inputs and config. Consist detects the fingerprint match and returns cached results instantly.


In [13]:
rerun_id = f"{SCENARIO_NAME}_{SESSION_ID}_baseline_rerun"
_ = run_scenario(land_use_config, demand_config, network_config, rerun_id)

rerun_runs = tracker.find_runs(parent_id=rerun_id, status="completed")
step_models = {"land_use_model", "demand_model", "network_model"}
step_runs = [run for run in rerun_runs if run.model_name in step_models]

cache_hits = [run for run in step_runs if (run.meta or {}).get("cache_hit")]
computed = [run for run in step_runs if not (run.meta or {}).get("cache_hit")]
print(f"Cache hits: {len(cache_hits)}, Recomputed: {len(computed)}")

pd.DataFrame(
    {
        "model": [run.model_name for run in step_runs],
        "status": [
            "cache_hit" if (run.meta or {}).get("cache_hit") else "recomputed"
            for run in step_runs
        ],
    }
).sort_values("model")

Cache hits: 3, Recomputed: 0


,model,status
1,demand_model,cache_hit
2,land_use_model,cache_hit
0,network_model,cache_hit


## Summary

This workflow demonstrates three core Consist capabilities:

1. **Scenario diff**: `tracker.diff_runs()` shows exactly what changed between any two runs - no README archaeology, no folder-name decoding.
2. **Automatic lineage**: `tracker.print_lineage()` traces any output back through the full computation chain. Automatically maintained - no manual DAG management.
3. **Smart caching**: Identical runs return cached results instantly. Config or input changes automatically invalidate downstream steps.

These patterns scale from simple linear pipelines to complex iterative workflows like full BEAM CORE. See `examples/02_iterative_workflows.ipynb` for the iterative case.

**Getting started:** Five lines of code per step. See [Consist on GitHub](https://github.com/LBNL-UCB-STI/consist).


## Querying Run History

Every run is recorded in a queryable database. `tracker.history()` returns a DataFrame of all runs — IDs, timing, status, tags — without writing any additional code. This is also available from the command line via `consist runs` for anyone who doesn't want to open a notebook.

In [14]:
tracker.history()[
    [
        "id",
        "parent_run_id",
        "status",
        "model_name",
        "description",
        "year",
        "iteration",
        "tags",
        "config_hash",
        "git_hash",
        "input_hash",
        "signature",
        "started_at",
        "ended_at",
        "created_at",
        "updated_at",
        "duration_seconds",
    ]
]

,id,parent_run_id,status,model_name,description,year,iteration,tags,config_hash,git_hash,input_hash,signature,started_at,ended_at,created_at,updated_at,duration_seconds
0,beam_core_demo_demo_baseline_rerun_network_mod...,beam_core_demo_demo_baseline_rerun,completed,network_model,None,None,None,[],36c622132fe98db9b79561e0092a2249e8512538b2a83b...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,c2a9ac48f09a220008c7edeaee1dd8e037640fd24539f0...,cb736cc79852eda0705e4a97babf689eab77ee03d0cf06...,2026-03-04 14:23:43.290987,2026-03-04 14:23:43.338221,2026-03-04 14:23:43.290987,2026-03-04 14:23:43.338221,0.047234
1,beam_core_demo_demo_baseline_rerun_demand_mode...,beam_core_demo_demo_baseline_rerun,completed,demand_model,None,None,None,[],c6e0d167bd66f1c35ecc4990f10cea1c7c0623d5cd43ba...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,3e8d3ddf1422f2d5b2f101e1d988dd483fa94b0c2e0df9...,328ac1ef00cbb3ccbb8ad9ce84b2035a418ede03649290...,2026-03-04 14:23:43.139680,2026-03-04 14:23:43.184235,2026-03-04 14:23:43.139680,2026-03-04 14:23:43.184235,0.044555
2,beam_core_demo_demo_baseline_rerun_land_use_mo...,beam_core_demo_demo_baseline_rerun,completed,land_use_model,None,None,None,[],bcb600c4e4761d9a185a4e1b2af829e64d7f7778e8ce6f...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,0ac09a2bfa29e63418dc8a9095ad676b19e0c0f0e6c1a4...,e27b394ae2abd6c86a3ad54f1d581c11a2e432b6700339...,2026-03-04 14:23:43.004700,2026-03-04 14:23:43.023154,2026-03-04 14:23:43.004700,2026-03-04 14:23:43.023154,0.018454
3,beam_core_demo_demo_baseline_rerun,NaN,completed,scenario,None,None,None,"[demo, beam_core, scenario_header]",e9cec66f5165d826bc29791d5317a9c6a8bec64052b814...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,0ac09a2bfa29e63418dc8a9095ad676b19e0c0f0e6c1a4...,2270a83ef72911a2fcbe198e8a645496ab1eeef8d35c2b...,2026-03-04 14:23:42.880367,2026-03-04 14:23:43.386047,2026-03-04 14:23:42.880367,2026-03-04 14:23:43.386047,0.505680
4,beam_core_demo_demo_high_parking_network_model...,beam_core_demo_demo_high_parking,completed,network_model,None,None,None,[],36c622132fe98db9b79561e0092a2249e8512538b2a83b...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,db4e1a469b2e2b04829edc90ba510c04b04ca9713f504c...,b63f3fd034640efaa6e8a3459b5432407be562b9b5242f...,2026-03-04 14:23:42.415804,2026-03-04 14:23:42.514561,2026-03-04 14:23:42.415804,2026-03-04 14:23:42.514561,0.098757
5,beam_core_demo_demo_high_parking_demand_model_...,beam_core_demo_demo_high_parking,completed,demand_model,None,None,None,[],dd7f566c61b733c1a5a9b1887f4eed5bb9d95fc1524844...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,3e8d3ddf1422f2d5b2f101e1d988dd483fa94b0c2e0df9...,f49032db27616904410357ef8e800e844fbfc053a09c3d...,2026-03-04 14:23:42.198692,2026-03-04 14:23:42.291749,2026-03-04 14:23:42.198692,2026-03-04 14:23:42.291749,0.093057
6,beam_core_demo_demo_high_parking_land_use_mode...,beam_core_demo_demo_high_parking,completed,land_use_model,None,None,None,[],bcb600c4e4761d9a185a4e1b2af829e64d7f7778e8ce6f...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,0ac09a2bfa29e63418dc8a9095ad676b19e0c0f0e6c1a4...,e27b394ae2abd6c86a3ad54f1d581c11a2e432b6700339...,2026-03-04 14:23:42.036114,2026-03-04 14:23:42.064164,2026-03-04 14:23:42.036114,2026-03-04 14:23:42.064164,0.028050
7,beam_core_demo_demo_high_parking,NaN,completed,scenario,None,None,None,"[demo, beam_core, scenario_header]",fe1211173f8695b99a61900d078f5ffec96cb131552edb...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,0ac09a2bfa29e63418dc8a9095ad676b19e0c0f0e6c1a4...,e593c88c64d60203ca9b151dac1bf053e2419f4ededa94...,2026-03-04 14:23:41.918077,2026-03-04 14:23:42.563477,2026-03-04 14:23:41.918077,2026-03-04 14:23:42.563477,0.645400
8,beam_core_demo_demo_baseline_network_model_a4f...,beam_core_demo_demo_baseline,completed,network_model,None,None,None,[],36c622132fe98db9b79561e0092a2249e8512538b2a83b...,c7543160e215f4404a7cf55ea34fd32021af92b8-dirty...,c2a9ac48f09a220008c7edeaee1dd8e037640fd24539f0...,cb736cc79852eda0705e4a97babf689e

  The same run history is available from the command line — no notebook required:

  ![consist runs CLI](img/beam_core_demo_runs.png)